# Neural Network from Scratch (NumPy Only)
Implements forward propagation, backpropagation, and gradient descent using only NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=5, random_state=42)
X = X.astype(np.float64)
y = y.reshape(-1, 1).astype(np.float64)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print('Data ready:', X_train.shape)

In [ ]:
# Activation functions
def sigmoid(z): return 1 / (1 + np.exp(-z))
def sigmoid_deriv(z): s = sigmoid(z); return s * (1 - s)
def relu(z): return np.maximum(0, z)
def relu_deriv(z): return (z > 0).astype(float)

# Weight initialization
def init_params(layer_dims):
    params = {}
    for l in range(1, len(layer_dims)):
        params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        params[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return params

print('Activation functions defined.')

In [ ]:
# Forward propagation
def forward_prop(X, params):
    cache = {'A0': X.T}
    L = len(params) // 2
    for l in range(1, L):
        Z = params[f'W{l}'] @ cache[f'A{l-1}'] + params[f'b{l}']
        A = relu(Z)
        cache[f'Z{l}'] = Z
        cache[f'A{l}'] = A
    ZL = params[f'W{L}'] @ cache[f'A{L-1}'] + params[f'b{L}']
    AL = sigmoid(ZL)
    cache[f'Z{L}'] = ZL
    cache[f'A{L}'] = AL
    return AL, cache

# Binary cross-entropy loss
def compute_loss(AL, y):
    m = y.shape[0]
    return -np.mean(y.T * np.log(AL + 1e-8) + (1 - y.T) * np.log(1 - AL + 1e-8))

print('Forward propagation defined.')

In [ ]:
# Backpropagation
def backward_prop(AL, y, params, cache):
    grads = {}
    m = y.shape[0]
    L = len(params) // 2
    dAL = -(y.T / (AL + 1e-8) - (1 - y.T) / (1 - AL + 1e-8))
    dZ = dAL * sigmoid_deriv(cache[f'Z{L}'])
    grads[f'dW{L}'] = (dZ @ cache[f'A{L-1}'].T) / m
    grads[f'db{L}'] = np.mean(dZ, axis=1, keepdims=True)
    dA = params[f'W{L}'].T @ dZ
    for l in reversed(range(1, L)):
        dZ = dA * relu_deriv(cache[f'Z{l}'])
        grads[f'dW{l}'] = (dZ @ cache[f'A{l-1}'].T) / m
        grads[f'db{l}'] = np.mean(dZ, axis=1, keepdims=True)
        if l > 1:
            dA = params[f'W{l}'].T @ dZ
    return grads

# Gradient descent update
def update_params(params, grads, lr):
    L = len(params) // 2
    for l in range(1, L+1):
        params[f'W{l}'] -= lr * grads[f'dW{l}']
        params[f'b{l}'] -= lr * grads[f'db{l}']
    return params

print('Backpropagation & gradient descent defined.')

In [ ]:
# Train the network
layer_dims = [5, 16, 8, 1]
params = init_params(layer_dims)
lr = 0.1
epochs = 500
loss_history = []

for epoch in range(epochs):
    AL, cache = forward_prop(X_train, params)
    loss = compute_loss(AL, y_train)
    loss_history.append(loss)
    grads = backward_prop(AL, y_train, params, cache)
    params = update_params(params, grads, lr)
    if (epoch+1) % 100 == 0:
        preds = (AL > 0.5).astype(int).T
        acc = np.mean(preds == y_train)
        print(f'Epoch {epoch+1} | Loss: {loss:.4f} | Train Acc: {acc:.4f}')

# Test accuracy
AL_test, _ = forward_prop(X_test, params)
preds_test = (AL_test > 0.5).astype(int).T
print(f'\nTest Accuracy: {np.mean(preds_test == y_test):.4f}')

In [ ]:
plt.plot(loss_history)
plt.title('Training Loss (NumPy From Scratch)')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.grid(True)
plt.show()